In [1]:
%pip install tensorflow==2.20.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 114.2 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.19.0
    Uninstalling tensorboard-2.19.0:
      Successfully uninstalled tensorboard-2.19.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.19.0
    Uninstalling tensorflow-2.19.0:
      Successfully uninstalled tensorflow-2.19.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-text 2.19.0 requires tensorflow<2.20,>=2.19.0, but you have tensorflow 2.20.0 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.20.0 which is incompatible.
tf-keras 2.19.0 requires tensorflow<2.20,>=2.19, but you have tensorflow 2.20.0 which is in

In [2]:
import kagglehub
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [3]:
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

dataset_path = kagglehub.dataset_download("shubham2703/five-crop-diseases-dataset")
print("Path to dataset files:", dataset_path)

data_dir = os.path.join(dataset_path, "Crop Diseases Dataset", "Crop Diseases", "Crop___Disease")
print("Data directory path:", data_dir)

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels='inferred',
    label_mode='categorical',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset='training',
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels='inferred',
    label_mode='categorical',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset='validation',
    seed=SEED
)

class_labels = train_ds.class_names
num_classes = len(class_labels)
print(f"Number of classes: {num_classes}")
print(f"Class labels: {class_labels}")

100%|██████████| 4.33G/4.33G [00:56<00:00, 83.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/shubham2703/five-crop-diseases-dataset/versions/1
Data directory path: /root/.cache/kagglehub/datasets/shubham2703/five-crop-diseases-dataset/versions/1/Crop Diseases Dataset/Crop Diseases/Crop___Disease
Found 13324 files belonging to 5 classes.
Using 10660 files for training.
Found 13324 files belonging to 5 classes.
Using 2664 files for validation.
Number of classes: 5
Class labels: ['Corn', 'Potato', 'Rice', 'Wheat', 'sugarcane']


In [4]:
def rescale(image, label):
    image = tf.cast(image, tf.float32)
    return image / 255.0, label

train_ds = train_ds.map(rescale)
val_ds = val_ds.map(rescale)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomContrast(0.2),
])

def apply_augmentation(image, label):
    return data_augmentation(image, training=True), label

In [5]:
train_ds = train_ds.map(apply_augmentation, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

base = ResNet50(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dropout(0.3)(x)
out = Dense(num_classes, activation='softmax')(x)
model = Model(base.input, out)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,597,957 (90.02 MB)

 Trainable params: 10,245 (40.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [6]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_plant_disease_model.h5', save_best_only=True)
]

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

Epoch 1/15
334/334 ━━━━━━━━━━━━━━━━━━━━ 0s 697ms/step - accuracy: 0.4028 - loss: 1.3939

334/334 ━━━━━━━━━━━━━━━━━━━━ 288s 814ms/step - accuracy: 0.4031 - loss: 1.3936 - val_accuracy: 0.6036 - val_loss: 1.0849
Epoch 2/15
333/334 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.5834 - loss: 1.1112

334/334 ━━━━━━━━━━━━━━━━━━━━ 49s 148ms/step - accuracy: 0.5834 - loss: 1.1111 - val_accuracy: 0.6629 - val_loss: 0.9794
Epoch 3/15
333/334 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.6178 - loss: 1.0375

334/334 ━━━━━━━━━━━━━━━━━━━━ 73s 121ms/step - accuracy: 0.6178 - loss: 1.0375 - val_accuracy: 0.7080 - val_loss: 0.9162
Epoch 4/15
333/334 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.6413 - loss: 0.9973

334/334 ━━━━━━━━━━━━━━━━━━━━ 40s 120ms/step - accuracy: 0.6413 - loss: 0.9973 - val_accuracy: 0.7188 - val_loss: 0.8709
Epoch 5/15
333/334 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.6619 - loss: 0.9588

...

334/334 ━━━━━━━━━━━━━━━━━━━━ 45s 136ms/step - accuracy: 0.7040 - loss: 0.8572 - val_accuracy: 0.7827 - val_loss: 0.7113
Epoch 12/15
333/334 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.7072 - loss: 0.8469

334/334 ━━━━━━━━━━━━━━━━━━━━ 84s 142ms/step - accuracy: 0.7072 - loss: 0.8468 - val_accuracy: 0.7872 - val_loss: 0.7015
Epoch 13/15
333/334 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.7089 - loss: 0.8359

334/334 ━━━━━━━━━━━━━━━━━━━━ 41s 123ms/step - accuracy: 0.7089 - loss: 0.8358 - val_accuracy: 0.7909 - val_loss: 0.6901
Epoch 14/15
333/334 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.7115 - loss: 0.8308

334/334 ━━━━━━━━━━━━━━━━━━━━ 82s 123ms/step - accuracy: 0.7116 - loss: 0.8308 - val_accuracy: 0.7954 - val_loss: 0.6752
Epoch 15/15
333/334 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.7100 - loss: 0.8196

334/334 ━━━━━━━━━━━━━━━━━━━━ 41s 122ms/step - accuracy: 0.7101 - loss: 0.8196 - val_accuracy: 0.7973 - val_loss: 0.6639


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
base.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks
)

model.save('plant_disease_cnn_final.h5')
print("Final model saved as 'plant_disease_cnn_final.h5'")
print("Best model saved as 'best_plant_disease_model.h5'")

Epoch 1/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 197s 406ms/step - accuracy: 0.9025 - loss: 3.5379 - val_accuracy: 0.3142 - val_loss: 21.4175
Epoch 2/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 104s 312ms/step - accuracy: 0.9941 - loss: 0.0253 - val_accuracy: 0.6862 - val_loss: 2.1086
Epoch 3/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 0s 292ms/step - accuracy: 0.9985 - loss: 0.0059

334/334 ━━━━━━━━━━━━━━━━━━━━ 224s 558ms/step - accuracy: 0.9985 - loss: 0.0059 - val_accuracy: 0.9756 - val_loss: 0.0972
Epoch 4/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 0s 292ms/step - accuracy: 0.9986 - loss: 0.0044

334/334 ━━━━━━━━━━━━━━━━━━━━ 194s 582ms/step - accuracy: 0.9986 - loss: 0.0044 - val_accuracy: 0.9947 - val_loss: 0.0201
Epoch 5/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 104s 311ms/step - accuracy: 0.9992 - loss: 0.0020 - val_accuracy: 0.9951 - val_loss: 0.0233


Final model saved as 'plant_disease_cnn_final.h5'
Best model saved as 'best_plant_disease_model.h5'
